# Проверка пересечений ИНН между выгрузками

Сравниваем уникальные ИНН между четырьмя источниками:
- **CRM** (`ECP_CRM_2022_2026.csv`) → колонка `inn_crm`
- **D_rating** (`D_rating.xlsx`) → колонка `inn_D`
- **Defaults** (`default_data.pkl`) → колонка `inn_def`
- **H2O** (`2023-2025_h20_data.xlsx`) → колонка `inn_h2o`

**Фильтр по дате:** 2023–2025 включительно.

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

DATA_DIR = "/home/jovyan/documents/DMUKP_FP_DEF/data"
DATE_FROM = pd.Timestamp("2023-01-01")
DATE_TO   = pd.Timestamp("2025-12-31")

## 1. Загрузка и подготовка данных

In [ ]:
# --- CRM ---
df_crm = pd.read_csv(
    f"{DATA_DIR}/ECP_CRM_2022_2026.csv",
    sep=";", encoding="utf-8-sig", dtype=str, low_memory=False,
)
df_crm.columns = df_crm.columns.str.strip()
df_crm = df_crm.rename(columns={
    'VAL.1': 'VAL_1', 'DESC_TEXT.1': 'DESC_TEXT_1', 'ROW_ID.1': 'ROW_ID_1',
    'AGREEMENT_NUM.1': 'AGREEMENT_NUM_1', 'ROW_ID.2': 'ROW_ID_2',
    'VAL.2': 'VAL_2', 'NAME.1': 'NAME_1', 'VAL.3': 'VAL_3', 'VAL.5': 'VAL_5',
})
df_crm["IDENTIFICATION_DATE"] = pd.to_datetime(
    df_crm["IDENTIFICATION_DATE"], dayfirst=True, errors="coerce"
)
df_crm = df_crm[
    (df_crm["IDENTIFICATION_DATE"] >= DATE_FROM) &
    (df_crm["IDENTIFICATION_DATE"] <= DATE_TO)
].copy()
df_crm = df_crm.rename(columns={"X_INN": "inn_crm"})
df_crm["inn_crm"] = df_crm["inn_crm"].astype(str).str.strip()

print(f"CRM (2023–2025): {len(df_crm):,} строк")
print(f"Уникальных inn_crm: {df_crm['inn_crm'].nunique():,}")

# --- D_rating ---
df_rating = pd.read_excel(f"{DATA_DIR}/D_rating.xlsx", dtype={"inn": str})
df_rating.columns = df_rating.columns.str.strip()
df_rating["confirmedat"] = pd.to_datetime(
    df_rating["confirmedat"], dayfirst=True, errors="coerce"
)
df_rating = df_rating[
    (df_rating["confirmedat"] >= DATE_FROM) &
    (df_rating["confirmedat"] <= DATE_TO)
].copy()
# inn может быть float64 — приводим к str через int, убирая .0
df_rating = df_rating.rename(columns={"inn": "inn_D"})
df_rating["inn_D"] = df_rating["inn_D"].apply(
    lambda x: str(int(float(x))) if pd.notna(x) else None
)

print(f"\nD_rating (2023–2025): {len(df_rating):,} строк")
print(f"Уникальных inn_D: {df_rating['inn_D'].nunique():,}")

# --- Defaults ---
df_def = pd.read_pickle(f"{DATA_DIR}/default_data.pkl")
df_def = df_def.astype(str).replace("nan", np.nan)
df_def.columns = df_def.columns.str.strip()
df_def = df_def.rename(columns={"start": "start_date", "cure": "cure_date", "finish": "finish_date"})
df_def["start_date"] = pd.to_datetime(df_def["start_date"], dayfirst=True, errors="coerce")
df_def = df_def[
    (df_def["start_date"] >= DATE_FROM) &
    (df_def["start_date"] <= DATE_TO)
].copy()
df_def = df_def.rename(columns={"inn": "inn_def"})
df_def["inn_def"] = df_def["inn_def"].astype(str).str.strip()

print(f"\nDefaults (2023–2025): {len(df_def):,} строк")
print(f"Уникальных inn_def: {df_def['inn_def'].nunique():,}")

# --- H2O ---
df_h2o = pd.read_excel(f"{DATA_DIR}/2023-2025_h20_data.xlsx", dtype=str)
df_h2o.columns = df_h2o.columns.str.strip()
df_h2o = df_h2o.rename(columns={
    'ИНН': 'inn_h2o', 'Наименование клиента': 'client_name',
    'CDI ID клиента': 'cdi_id', 'ОПФ': 'legal_form', 'РФ': 'rf',
    'Сегмент': 'segment', 'Вид мониторинга': 'mon_type',
    'ИД справочного фактора в CRM': 'h20_fp_id',
    'Наименование фактора проблемности': 'fp_name',
    'Дата выявления фактора проблемности': 'fp_start_date',
    'Номер фактора проблемности из справочника': 'fp_number',
})
df_h2o["fp_start_date"] = pd.to_datetime(
    df_h2o["fp_start_date"], dayfirst=True, errors="coerce"
)
df_h2o = df_h2o[
    (df_h2o["fp_start_date"] >= DATE_FROM) &
    (df_h2o["fp_start_date"] <= DATE_TO)
].copy()
df_h2o["inn_h2o"] = df_h2o["inn_h2o"].astype(str).str.strip()

print(f"\nH2O (2023–2025): {len(df_h2o):,} строк")
print(f"Уникальных inn_h2o: {df_h2o['inn_h2o'].nunique():,}")

## 2. Уникальные ИНН и пересечения

In [ ]:
inn_crm = set(df_crm["inn_crm"].dropna().unique())
inn_D   = set(df_rating["inn_D"].dropna().unique())
inn_def = set(df_def["inn_def"].dropna().unique())
inn_h2o = set(df_h2o["inn_h2o"].dropna().unique())

print("=" * 60)
print("УНИКАЛЬНЫЕ ИНН В КАЖДОЙ ВЫГРУЗКЕ (2023–2025)")
print("=" * 60)
print(f"  CRM:        {len(inn_crm):>6,}")
print(f"  D_rating:   {len(inn_D):>6,}")
print(f"  Defaults:   {len(inn_def):>6,}")
print(f"  H2O:        {len(inn_h2o):>6,}")

print(f"\n{'=' * 60}")
print("ПЕРЕСЕЧЕНИЯ МЕЖДУ ПАРАМИ")
print("=" * 60)
crm_and_D   = inn_crm & inn_D
crm_and_def = inn_crm & inn_def
crm_and_h2o = inn_crm & inn_h2o
D_and_def   = inn_D & inn_def
D_and_h2o   = inn_D & inn_h2o
def_and_h2o = inn_def & inn_h2o

print(f"  CRM  ∩  D_rating:     {len(crm_and_D):>6,}")
print(f"  CRM  ∩  Defaults:     {len(crm_and_def):>6,}")
print(f"  CRM  ∩  H2O:          {len(crm_and_h2o):>6,}")
print(f"  D_rating ∩ Defaults:  {len(D_and_def):>6,}")
print(f"  D_rating ∩ H2O:       {len(D_and_h2o):>6,}")
print(f"  Defaults ∩ H2O:       {len(def_and_h2o):>6,}")

all_four = inn_crm & inn_D & inn_def & inn_h2o
print(f"\n  Во всех четырёх:      {len(all_four):>6,}")

only_crm = inn_crm - inn_D - inn_def - inn_h2o
only_D   = inn_D - inn_crm - inn_def - inn_h2o
only_def = inn_def - inn_crm - inn_D - inn_h2o
only_h2o = inn_h2o - inn_crm - inn_D - inn_def

print(f"\n{'=' * 60}")
print("ТОЛЬКО В ОДНОЙ ВЫГРУЗКЕ")
print("=" * 60)
print(f"  Только в CRM:       {len(only_crm):>6,}")
print(f"  Только в D_rating:  {len(only_D):>6,}")
print(f"  Только в Defaults:  {len(only_def):>6,}")
print(f"  Только в H2O:       {len(only_h2o):>6,}")

## 3. Сводная таблица пересечений

In [ ]:
summary = pd.DataFrame([
    {"Группа": "CRM (всего уник. ИНН)", "Кол-во": len(inn_crm)},
    {"Группа": "D_rating (всего уник. ИНН)", "Кол-во": len(inn_D)},
    {"Группа": "Defaults (всего уник. ИНН)", "Кол-во": len(inn_def)},
    {"Группа": "H2O (всего уник. ИНН)", "Кол-во": len(inn_h2o)},
    {"Группа": "—", "Кол-во": "—"},
    {"Группа": "CRM ∩ D_rating", "Кол-во": len(crm_and_D)},
    {"Группа": "CRM ∩ Defaults", "Кол-во": len(crm_and_def)},
    {"Группа": "CRM ∩ H2O", "Кол-во": len(crm_and_h2o)},
    {"Группа": "D_rating ∩ Defaults", "Кол-во": len(D_and_def)},
    {"Группа": "D_rating ∩ H2O", "Кол-во": len(D_and_h2o)},
    {"Группа": "Defaults ∩ H2O", "Кол-во": len(def_and_h2o)},
    {"Группа": "Во всех четырёх", "Кол-во": len(all_four)},
    {"Группа": "—", "Кол-во": "—"},
    {"Группа": "Только в CRM", "Кол-во": len(only_crm)},
    {"Группа": "Только в D_rating", "Кол-во": len(only_D)},
    {"Группа": "Только в Defaults", "Кол-во": len(only_def)},
    {"Группа": "Только в H2O", "Кол-во": len(only_h2o)},
])

display(summary.style.hide(axis="index"))

## 4. Примеры ИНН

In [ ]:
N_EXAMPLES = 5

# --- ИНН во всех четырёх выгрузках ---
print("=" * 60)
print(f"ИНН, присутствующие во всех четырёх выгрузках (до {N_EXAMPLES} примеров):")
print("=" * 60)
examples_all = sorted(all_four)[:N_EXAMPLES]
if examples_all:
    for inn in examples_all:
        n_crm = len(df_crm[df_crm["inn_crm"] == inn])
        n_rat = len(df_rating[df_rating["inn_D"] == inn])
        n_def = len(df_def[df_def["inn_def"] == inn])
        n_h2o = len(df_h2o[df_h2o["inn_h2o"] == inn])
        print(f"  ИНН {inn}  →  CRM: {n_crm}, D_rating: {n_rat}, Defaults: {n_def}, H2O: {n_h2o} строк")
else:
    print("  Нет ИНН, присутствующих во всех четырёх выгрузках.")

# --- ИНН только в D_rating ---
print(f"\n{'=' * 60}")
print(f"ИНН, присутствующие ТОЛЬКО в D_rating (до {N_EXAMPLES} примеров):")
print("=" * 60)
examples_only_D = sorted(only_D)[:N_EXAMPLES]
if examples_only_D:
    for inn in examples_only_D:
        rows = df_rating[df_rating["inn_D"] == inn]
        dates = rows["confirmedat"].dt.strftime("%d.%m.%Y").tolist()
        print(f"  ИНН {inn}  →  {len(rows)} строк, даты рейтинга: {', '.join(dates)}")
else:
    print("  Все ИНН из D_rating встречаются хотя бы в одной другой выгрузке.")

## 5. Матрица пересечений

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sources = {"CRM": inn_crm, "D_rating": inn_D, "Defaults": inn_def, "H2O": inn_h2o}
names = list(sources.keys())

# --- 1. Матрица попарных пересечений ---
matrix = pd.DataFrame(index=names, columns=names, dtype=int)
for a in names:
    for b in names:
        matrix.loc[a, b] = len(sources[a] & sources[b])

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(matrix.astype(int), annot=True, fmt=",", cmap="YlOrRd",
            linewidths=1, linecolor="white", ax=ax)
ax.set_title("Попарные пересечения ИНН (2023–2025)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

# --- 2. Столбчатая диаграмма: уникальные ИНН по выгрузкам ---
fig, ax = plt.subplots(figsize=(8, 4))
counts = [len(sources[n]) for n in names]
colors = ["#4C72B0", "#DD8452", "#55A868", "#C44E52"]
bars = ax.barh(names, counts, color=colors, edgecolor="white")
for bar, cnt in zip(bars, counts):
    ax.text(bar.get_width() + max(counts) * 0.01, bar.get_y() + bar.get_height() / 2,
            f"{cnt:,}", va="center", fontsize=11)
ax.set_xlabel("Уникальных ИНН")
ax.set_title("Количество уникальных ИНН по выгрузкам (2023–2025)", fontsize=13, fontweight="bold")
ax.set_xlim(0, max(counts) * 1.15)
plt.tight_layout()
plt.show()